In [3]:
import cv2
import os
import csv
from ultralytics import YOLO

# 1. Configuración inicial
ruta_pesos = '../models/entrenamiento_script_estable/weights/best.pt'
model = YOLO(ruta_pesos)
nombres_clases = {0: 'Vehiculo', 1: 'Peaton', 2: 'Dos_Ruedas', 3: 'Senales'}

os.makedirs('../reports/alertas_visuales', exist_ok=True)

ruta_video_entrada = '../test/MedioDiaLluvia.mov' 
ruta_video_salida = '../reports/alertas_visuales/MedioDiaLluvia_alertas.mp4'
ruta_csv_salida = '../reports/alertas_visuales/MedioDiaLluvia_registro_alertas.csv' # <--- NUEVO ARCHIVO

cap = cv2.VideoCapture(ruta_video_entrada)

if not cap.isOpened():
    print(f"Error: No se pudo abrir el video en {ruta_video_entrada}")
else:
    ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    area_total = ancho * alto

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(ruta_video_salida, fourcc, fps, (ancho, alto))

    print(f"Iniciando procesamiento y registro de datos: {total_frames} frames a {fps} FPS...")

    frame_count = 0

    # --- NUEVO: ABRIR EL ARCHIVO CSV EN MODO ESCRITURA ---
    with open(ruta_csv_salida, mode='w', newline='', encoding='utf-8') as archivo_csv:
        escritor_csv = csv.writer(archivo_csv)
        # Escribir los encabezados de las columnas
        escritor_csv.writerow(['Frame', 'Tiempo (Segundos)', 'Objeto', 'Confianza (%)', 'Nivel de Riesgo'])

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break 

            resultados = model(frame, verbose=False)
            
            # Calcular el segundo exacto del video
            tiempo_actual_segundos = frame_count / fps

            for resultado in resultados:
                cajas = resultado.boxes
                for caja in cajas:
                    confianza = float(caja.conf[0])
                    if confianza < 0.4:
                        continue
                        
                    clase_id = int(caja.cls[0])
                    nombre_objeto = nombres_clases[clase_id]
                    x1, y1, x2, y2 = map(int, caja.xyxy[0].tolist())
                    
                    area_obj = (x2 - x1) * (y2 - y1)
                    porcentaje = (area_obj / area_total) * 100
                    
                    color = (0, 255, 0) 
                    texto_alerta = f"{nombre_objeto} {porcentaje:.1f}%"
                    nivel_riesgo_actual = "Bajo" # Por defecto
                    
                    if clase_id == 0 and porcentaje > 15.0:
                        color = (0, 0, 255) 
                        texto_alerta = "¡COLISION INMINENTE!"
                        nivel_riesgo_actual = "ALTO"
                    elif clase_id in [1, 2] and porcentaje > 5.0:
                        color = (0, 0, 255) 
                        texto_alerta = "¡PELIGRO PEATON/MOTO!"
                        nivel_riesgo_actual = "CRÍTICO"
                    elif porcentaje > 8.0:
                        color = (0, 255, 255) 
                        texto_alerta = "Precaucion"
                        nivel_riesgo_actual = "MEDIO"

                    # --- NUEVO: REGISTRAR EN EL CSV SOLO SI HAY RIESGO ---
                    if nivel_riesgo_actual in ["MEDIO", "ALTO", "CRÍTICO"]:
                        escritor_csv.writerow([
                            frame_count, 
                            round(tiempo_actual_segundos, 2), 
                            nombre_objeto, 
                            round(confianza * 100, 1), 
                            nivel_riesgo_actual
                        ])

                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                    (w, h), _ = cv2.getTextSize(texto_alerta, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
                    cv2.rectangle(frame, (x1, y1 - 20), (x1 + w, y1), color, -1)
                    cv2.putText(frame, texto_alerta, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1)

            out.write(frame)
            frame_count += 1
            
            if frame_count % 50 == 0:
                print(f"Procesando frame {frame_count}/{total_frames}...")

    cap.release()
    out.release()
    cv2.destroyAllWindows()

    print(f"¡Procesamiento completado!")
    print(f"Video guardado en: {ruta_video_salida}")
    print(f"Reporte de Alertas CSV guardado en: {ruta_csv_salida}")

Iniciando procesamiento y registro de datos: 1203 frames a 30 FPS...
Procesando frame 50/1203...
Procesando frame 100/1203...
Procesando frame 150/1203...
Procesando frame 200/1203...
Procesando frame 250/1203...
Procesando frame 300/1203...
Procesando frame 350/1203...
Procesando frame 400/1203...
Procesando frame 450/1203...
Procesando frame 500/1203...
Procesando frame 550/1203...
Procesando frame 600/1203...
Procesando frame 650/1203...
Procesando frame 700/1203...
Procesando frame 750/1203...
Procesando frame 800/1203...
Procesando frame 850/1203...
Procesando frame 900/1203...
Procesando frame 950/1203...
Procesando frame 1000/1203...
Procesando frame 1050/1203...
Procesando frame 1100/1203...
Procesando frame 1150/1203...
Procesando frame 1200/1203...
¡Procesamiento completado!
Video guardado en: ../reports/alertas_visuales/MedioDiaLluvia_alertas.mp4
Reporte de Alertas CSV guardado en: ../reports/alertas_visuales/MedioDiaLluvia_registro_alertas.csv
